In [1]:
%pip install --upgrade --quiet google-genai google-auth pandas

Note: you may need to restart the kernel to use updated packages.


In [4]:
import datetime

from google import genai
from google.genai.types import (
    CreateCachedContentConfig,
    EmbedContentConfig,
    FunctionDeclaration,
    GenerateContentConfig,
    HarmBlockThreshold,
    HarmCategory,
    Part,
    SafetySetting,
    Tool,
)

In [ ]:
import google.auth
import os

# Detect the project ID from the Workbench instance instead of hard-coding it
_, project = google.auth.default()
project = project or os.environ.get("GOOGLE_CLOUD_PROJECT")
location = "global"
print("Using project:", project)

In [2]:
MODEL_ID = "gemini-3.1-flash-lite"

In [5]:
client = genai.Client(vertexai = True, project = project, location = location)

In [6]:
chat = client.chats.create(
    model = MODEL_ID
)

In [7]:
response = chat.send_message("Tell me about the history of mighals in 300 words")
print(response.text)

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'Publisher model `projects/YOUR_PROJECT_ID/locations/us-central1/publishers/google/models/gemini-2.0-flash-lite` was not found or your project does not have access to it. Ensure you are using a valid model name and that the model is available in the specified region. For more information, see: https://docs.cloud.google.com/gemini-enterprise-agent-platform/resources/locations.', 'status': 'NOT_FOUND'}}

In [ ]:
response = chat.send_message("Tell me about the history of the Jahangir in 300 words")
print(response.text)

In [ ]:
response = chat.get_history()
print(response)

In [7]:
#Test to see my genai client is connected or not

In [ ]:
# Whenever I am activating my instance I always have to run a generate_content
# to check the client is working or not. That means I am losing tokens just to test that its working or not. Tell me a best possible way to check if the client is working meaning the SDK is ready to generate response without error and also I want to use less tokens


In [8]:
response = client.models.count_tokens(
    model=MODEL_ID,
    contents="test"
)
print(f"Client is ready! Token count: {response.total_tokens}")

Client is ready! Token count: 1


In [6]:
response = client.models.generate_content(
    model = MODEL_ID,
    contents = "Tell me about the summer weather"
)
print(response.text)

Because summer weather varies drastically depending on where you are on the planet, it is best described through a few different lenses: geography, atmosphere, and human experience.

Here is a breakdown of what "summer weather" generally entails:

### 1. The Hemispherical Divide
*   **Northern Hemisphere:** Summer officially begins with the Summer Solstice (around June 21st) and lasts until the Autumnal Equinox (around September 22nd).
*   **Southern Hemisphere:** Summer begins with the Summer Solstice (around December 21st) and lasts until the Vernal Equinox (around March 20th).
*   **The Equator:** Locations near the equator do not experience "summer" in the traditional sense. Instead, they often have "wet" and "dry" seasons, and temperatures remain consistently warm year-round.

### 2. Meteorological Characteristics
Regardless of the location, summer is defined by the tilt of the Earth’s axis, which causes one hemisphere to be tilted toward the sun. This results in:
*   **Longer Day

In [ ]:
# changing the structure of the output

In [16]:
from google.genai import types
from pydantic import BaseModel

In [17]:
class TripDetails(BaseModel):
    City : str
    Historic_sites : list[str]
    Indian_food_places_nearby : list[str]
    Weather_today_and_date : str
    Best_time_to_visit : str


prompt = "Plan a trip for me to go to Sandiego and give me the details"

system_instruction = """
You are a wonderful tour guide. Whenever you get a place and you want to tour the places that is not common to the people. That is the quality that makes you unique.
Because most tour guides will only take to the popular places so you hold an USP of showing unique places that is undiscovered to the tourists. Whenever giving any 
recommendation just add one more line for the tourists to know what makes the p[lace special"""

response = client.models.generate_content(
    model = MODEL_ID,
    contents = prompt,
    config = GenerateContentConfig(
        response_mime_type = "application/json",
        system_instruction = system_instruction,
        response_schema = TripDetails,
    ),
)

print(response.text)


ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'Publisher model `projects/YOUR_PROJECT_ID/locations/us-central1/publishers/google/models/gemini-1.5-flash` was not found or your project does not have access to it. Ensure you are using a valid model name and that the model is available in the specified region. For more information, see: https://docs.cloud.google.com/gemini-enterprise-agent-platform/resources/locations.', 'status': 'NOT_FOUND'}}

In [25]:
# using Tools pre defined ones

In [27]:
tools = [Tool(google_search = types.GoogleSearch())]

In [30]:
response = client.models.generate_content(
    model = MODEL_ID,
    contents = "Tell me about the latest model in hugging face that has the highest number fo stars.",
    config = GenerateContentConfig(
        tools = tools,
        temperature = 0.1,
    ),
)

print(response.text)

On Hugging Face, the term "stars" is typically referred to as **"likes"** for models. While there is no single "latest" model that holds the all-time record, the platform's most popular models are generally foundational architectures that have been established for years.

### Key Points Regarding "Most Starred" Models
*   **Likes vs. Stars:** On the Hugging Face Hub, users "like" models. The term "stars" is primarily used for repositories on GitHub (such as the [Hugging Face `transformers` library](https://github.com/huggingface/transformers), which has over 160,000 stars).
*   **Most Popular Models:** The models with the highest number of likes are typically foundational, widely used models that serve as the basis for countless other projects. These include:
    *   **`sentence-transformers/all-MiniLM-L6-v2`**: Frequently cited as one of the most liked and downloaded models for sentence similarity tasks.
    *   **`bert-base-uncased`**: A foundational encoder that remains one of the m

In [31]:
#Using function declaration as a custom function to the tools

In [47]:
hugging_face_models = FunctionDeclaration(
    name = "hugging_face_models",
    description = "",
    parameters = {
        "type" : "OBJECT",
        "properties" : {
            "models_list" : {
                "type" : "ARRAY",
                "description" : "A list of top models requested by user",
                "items" : {
                    "type" : "OBJECT",
                    "properties" : {
                        "model_name" : {
                            "type" : "STRING",
                            "description" : "model found from the results"
                        },
                        "description" : {
                            "type" : "STRING",
                            "description" : "one line for each models"
                        },
                        "date_developed" : {
                            "type" : "STRING",
                            "description" : "when was the model developed or published to hugging face"
                        },
                },
                "required" : ["model_name", "description", "date_developed"]
            }
        }
        },
        "required" : ["models_list"]  
    }
)

get_models_tool = Tool(
    function_declarations=[hugging_face_models],)

In [51]:
import json

In [53]:
response = client.models.generate_content(
    model = MODEL_ID,
    contents = "Get the top 5 most used models in hugging face. Add one line description to it. Also list when was the model created",
    config = GenerateContentConfig(
        tools = [get_models_tool],
        temperature = 0.1,
    ),
)

args_dict = response.function_calls[0].args
print(json.dumps(args_dict, indent= 4))

{
    "models_list": [
        {
            "description": "A powerful, open-weights large language model developed by OpenAI (via Meta's LLaMA architecture influence) and widely used for fine-tuning.",
            "date_developed": "2022-11-30",
            "model_name": "meta-llama/Llama-2-7b-chat-hf"
        },
        {
            "date_developed": "2023-07-18",
            "model_name": "runwayml/stable-diffusion-v1-5",
            "description": "A highly efficient and popular model for text-to-image generation tasks."
        },
        {
            "date_developed": "2022-05-26",
            "description": "A versatile model for automatic speech recognition (ASR) and speech translation.",
            "model_name": "openai/whisper-large-v2"
        },
        {
            "description": "A widely used model for generating high-quality embeddings for text retrieval and semantic search.",
            "model_name": "sentence-transformers/all-MiniLM-L6-v2",
            "date_dev